In [ ]:
conda create -u "dfu-venv" python=3.12 -y


In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

# 1. State 정의
class GraphState(TypedDict):
    question: str
    documents: List[str]
    answer: str


llm = ChatOpenAI(model="gpt-4o-mini")


# 2. Retriever node
def retrieve(state: GraphState):
    question = state["question"]

    # 실제 환경에서는 vector DB 검색
    docs = [
        "Diabetic foot ulcers are wounds occurring in diabetic patients.",
        "Treatment requires infection control and wound care."
    ]

    return {"documents": docs}


# 3. LLM node
def generate(state: GraphState):
    question = state["question"]
    docs = state["documents"]

    context = "\n".join(docs)

    prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)

    return {"answer": response.content}


# 4. Graph 생성
workflow = StateGraph(GraphState)

workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)

workflow.set_entry_point("retrieve")

workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()


# 5. 실행
result = app.invoke({
    "question": "What is diabetic foot ulcer?"
})

print(result["answer"])

: 